In [ ]:
!pip install -U pip
!pip install -U torch torchvision transformers datasets bitsandbytes accelerate peft
!sudo apt-get install -y cmake

In [ ]:
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
!nvidia-smi

In [2]:
import torch
import transformers
import datasets
import bitsandbytes
import accelerate
import peft
import unsloth

torch.__version__, transformers.__version__, datasets.__version__, bitsandbytes.__version__, accelerate.__version__, peft.__version__

2025-02-23 19:40:30.245055: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-23 19:40:30.259025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-23 19:40:30.276974: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-23 19:40:30.282422: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-23 19:40:30.295513: I tensorflow/core/platform/cpu_feature_guar

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


('2.6.0+cu124', '4.49.0', '3.3.2', '0.45.2', '1.4.0', '0.14.0')

# Get the model

In [3]:
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # 4bit quantization to reduce memory usage

In [3]:
from unsloth import FastLanguageModel

model_names = [
    "unsloth/Phi-4",  # Phi-4 2x faster!
    "unsloth/Phi-4-unsloth-bnb-4bit",  # Phi-4 Unsloth Dynamic 4-bit Quant
]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_names[0],
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.568 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(100352, 5120, padding_idx=100351)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (k_proj): Linear4bit(in_features=5120, out_features=1280, bias=False)
          (v_proj): Linear4bit(in_features=5120, out_features=1280, bias=False)
          (o_proj): Linear4bit(in_features=5120, out_features=5120, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=5120, out_features=17920, bias=False)
          (up_proj): Linear4bit(in_features=5120, out_features=17920, bias=False)
          (down_proj): Linear4bit(in_features=17920, out_features=5120, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((512

# Inference with the pretrained model

In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

user_query = "I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?"
messages = [
    {"role": "user", "content": user_query},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True,
                         temperature=1.5, min_p=0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


["<|im_start|>user<|im_sep|>I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|im_end|><|im_start|>assistant<|im_sep|>Incorporating ground lean meat into your breakfast can be a healthy choice, especially if you're aiming to lose weight and improve your fitness. Lean meats are a good source of protein, which can help you feel full and satisfied, potentially reducing overall calorie intake. Here are a few considerations to ensure it fits well into your weight loss and health goals:\n\n1. **Portion Control**: Even lean meats should be consumed in moderation. A typical serving size is about 3-4 ounces (85-113 grams).\n\n2. **Balanced Meal**: Pair your lean meat with vegetables and whole grains to create a balanced meal. This will provide fiber, vitamins, and minerals that support overall health.\n\n3. **Cooking Method**: Opt for healthier cooking

# Set LoRa adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRa rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,  # Rank Stabilized LoRA
    loftq_config=None,  # LoftQ
)

Unsloth 2025.2.15 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [6]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(100352, 5120, padding_idx=100351)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

# Get the dataset

In [7]:
from datasets import load_dataset

dataset = load_dataset("Tom158/Nutritional-LLama", split="train")

In [8]:
dataset.column_names

['System', 'User', 'Nutritionist', 'text']

In [9]:
dataset[0]["System"]

'You serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.'

In [10]:
dataset[0]["User"]

"I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?"

In [11]:
dataset[0]["Nutritionist"]

"Ground lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime."

In [12]:
dataset[0]["text"]

"<s>[INST] <<SYS>>\nYou serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.\n<</SYS>>\n\nI've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight? [/INST] Ground lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime. </s>"

In [13]:
dataset = dataset.rename_column("text", "text_llama2")

In [14]:
dataset.column_names

['System', 'User', 'Nutritionist', 'text_llama2']

In [15]:
tokenizer.eos_token

'<|im_end|>'

In [16]:
def format_prompt(example):
    system = example["System"]
    user = example["User"]
    assistant = example["Nutritionist"]

    llama3_1_prompt = f"""\
<|im_start|>system<|im_sep|>
{system}<|im_end|>
<|im_start|>user<|im_sep|>
{user}<|im_end|>
<|im_start|>assistant<|im_sep|>
{assistant}<|im_end|>\
"""
    return {"text": llama3_1_prompt}
    

dataset = dataset.map(format_prompt)

In [17]:
dataset.column_names

['System', 'User', 'Nutritionist', 'text_llama2', 'text']

In [18]:
dataset[0]["text"]

"<|im_start|>system<|im_sep|>\nYou serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.<|im_end|>\n<|im_start|>user<|im_sep|>\nI've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|im_end|>\n<|im_start|>assistant<|im_sep|>\nGround lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime.<|im_end|>"

# Train the model

In [19]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # num_train_epochs=1, # Set this for 1 full training run.
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="steps",
        save_steps=50,
        report_to="none",
    ),
)

Converting train dataset to ChatML (num_proc=2):   0%|          | 0/1488 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/1488 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1488 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1488 [00:00<?, ? examples/s]

In [20]:
# Mask train on the assistant outputs and ignore the loss on the user's inputs
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user<|im_sep|>",
    response_part="<|im_start|>assistant<|im_sep|>",
)

Map:   0%|          | 0/1488 [00:00<?, ? examples/s]

In [21]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

"<|im_start|>system<|im_sep|>You serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.<|im_end|><|im_start|>user<|im_sep|>I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|im_end|><|im_start|>assistant<|im_sep|>Ground lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime.<|im_end|>"

In [22]:
space = tokenizer(" ", add_special_tokens=False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

"                                                                                          Ground lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime.<|im_end|>"

In [23]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.568 GB.
10.0 GB of memory reserved.


In [24]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,488 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 65,536,000


Step,Training Loss
1,1.756300
2,1.765700
3,1.738500
4,1.187300
5,0.291400
6,0.039900
7,0.058000
8,0.002400
9,0.038100
10,0.025600


In [25]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

645.1702 seconds used for training.
Peak reserved memory = 12.557 GB.
Peak reserved memory for training = 2.557 GB.
Peak reserved memory % of max memory = 86.196 %.
Peak reserved memory for training % of max memory = 17.552 %.


# Save the fine-tuned model

In [27]:
model.save_pretrained("nutritionalist-phi4")
tokenizer.save_pretrained("nutritionalist-phi4")

('nutritionalist-phi4/tokenizer_config.json',
 'nutritionalist-phi4/special_tokens_map.json',
 'nutritionalist-phi4/vocab.json',
 'nutritionalist-phi4/merges.txt',
 'nutritionalist-phi4/added_tokens.json',
 'nutritionalist-phi4/tokenizer.json')

In [28]:
model.push_to_hub("noroozi/nutritionalist-phi4", token=HF_TOKEN)
tokenizer.push_to_hub("noroozi/nutritionalist-phi4", token=HF_TOKEN)

README.md:   0%|          | 0.00/580 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/262M [00:00<?, ?B/s]

Saved model to https://huggingface.co/noroozi/nutritionalist-phi4


### gguf

In [ ]:
model.save_pretrained_gguf("nutritionalist-phi4-gguf", tokenizer, 
                           quantization_method=["q4_k_m", "q8_0", "q5_k_m",])

In [ ]:
model.push_to_hub_gguf(
        "noroozi/nutritionalist-phi4-gguf",
        tokenizer,
        quantization_method=["q4_k_m", "q8_0", "q5_k_m",],
        token=HF_TOKEN,
    )

# Load the model

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="nutritionalist-phi4",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.568 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Will load nutritionalist-phi4 as a legacy tokenizer.
Unsloth 2025.2.15 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


# Inference

In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

user_query = "I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?"
messages = [
    {"role": "user", "content": user_query},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True,
                         temperature=1.5, min_p=0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Unsloth: Not a fast tokenizer, so can't process it as of yet :(
Please log a Github issue if you want this as a new feature!
Your chat template will still work, but it won't add or edit tokens.


["<|im_start|> user <|im_sep|> I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|im_end|> <|im_start|> assistant <|im_sep|> I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|im_end|>"]

In [7]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=512,
                   use_cache=True, temperature=1.5, min_p=0.1)

I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|im_end|>


# Ollama Inference

In [ ]:
print(tokenizer._ollama_modelfile)

In [ ]:
!ollama create nutritionalist-phi4 -f ./nutritionalist-phi4-gguf/Modelfile

In [ ]:
!curl http://localhost:11434/api/chat -d '{ \
    "model": "nutritionalist-phi4", \
    "messages": [ \
        { "role": "user", "content": "I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?" } \
    ] \
    }'